# ZH-VI Machine Translation — Unified Transformer Pipeline
**Architecture:** Encoder-Decoder Transformer  
- Grouped-Query Attention (GQA) + RoPE  
- SwiGLU FFN · RMSNorm  
- Contrastive Learning (fine-tune stage)  

**Pipeline:** Data Preparation → Tokenizer → Training → Contrastive Fine-Tune → Inference


## Cell 1 — Install Dependencies & Imports

In [1]:
# Install required packages (uncomment if needed)
!pip install -qq sacrebleu sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
import math
import random
import shutil
import tempfile
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")               # headless backend; swap to "TkAgg" if interactive
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from tqdm import tqdm

try:
    import sacrebleu as sacrebleu_lib
    SACREBLEU_AVAILABLE = True
except ImportError:
    SACREBLEU_AVAILABLE = False
    warnings.warn("sacrebleu not found — BLEU scores will be 0.0")

print("✅ All imports successful.")
print(f"   PyTorch version : {torch.__version__}")
print(f"   CUDA available  : {torch.cuda.is_available()}")


✅ All imports successful.
   PyTorch version : 2.10.0+cu128
   CUDA available  : True


## Cell 2 — Global Seeds & Device Setup

In [3]:
# ── Global Seeds ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device          : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"[INFO] GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"[INFO] GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


[INFO] Device          : cuda
[INFO] GPU name        : Tesla T4
[INFO] GPU memory      : 15.6 GB


## Cell 3 — Configuration (`TranslationConfig`)
Centralised dataclass holding **all** hyper-parameters for the pipeline.

In [4]:
# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================

@dataclass
class TranslationConfig:
    # Paths
    base_dir:           str = "."
    train_zh_file:      str = "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/train/train.zh"
    train_vi_file:      str = "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/train/train.vi"
    tokenizer_dir:      str = "./tokenizer_train32"
    clean_data_dir:     str = "./clean_data"
    checkpoint_dir:     str = "./checkpoints"
    contrastive_dir:    str = "./checkpoints_contrastive"
    plot_dir:           str = "./plots"

    # Tokenizer
    vocab_size:         int   = 8_000
    max_len:            int   = 32

    # Model architecture
    d_model:            int   = 768
    n_heads:            int   = 12
    n_kv_heads:         int   = 4
    num_encoder_layers: int   = 8
    num_decoder_layers: int   = 8
    d_ff:               int   = 3_072
    dropout:            float = 0.01
    rope_base:          float = 10_000.0

    # Special tokens
    pad_token:  str = "<pad>"
    bos_token:  str = "<s>"
    eos_token:  str = "</s>"
    unk_token:  str = "<unk>"
    zh_token:   str = "<2zh>"
    vi_token:   str = "<2vi>"

    # Training
    batch_size:         int   = 128
    num_epochs:         int   = 40
    lr_base:            float = 2e-4
    warmup_steps:       int   = 200
    weight_decay:       float = 0.01
    label_smoothing:    float = 0.01
    grad_clip:          float = 1.0
    span_mask_prob:     float = 0.01
    vi2zh_epoch_ratio:  float = 0.70
    num_workers:        int   = 4
    save_every:         int   = 10
    val_split:          float = 0.05
    max_bleu_samples:   int   = 300

    # Contrastive fine-tune
    cl_proj_dim:            int   = 768
    cl_tau:                 float = 0.07
    cl_lambda_max:          float = 0.10
    cl_warmup_steps:        int   = 200
    cl_lr:                  float = 5e-5
    cl_num_epochs:          int   = 20
    cl_batch_size:          int   = 64
    cl_save_every:          int   = 5

    # Inference
    beam_size:          int   = 3
    top_k:              int   = 5
    length_penalty:     float = 0.6

    device: torch.device = field(default_factory=lambda: DEVICE)


cfg = TranslationConfig()

for d in [cfg.tokenizer_dir, cfg.clean_data_dir,
          cfg.checkpoint_dir, cfg.contrastive_dir, cfg.plot_dir]:
    os.makedirs(d, exist_ok=True)

TOKENIZER_PREFIX = os.path.join(cfg.tokenizer_dir, "spm_zh_vi_joint")

# ── Print configuration summary ───────────────────────────────────────────────
print("=" * 55)
print("  TranslationConfig Summary")
print("=" * 55)
print(f"  d_model          : {cfg.d_model}")
print(f"  n_heads / n_kv   : {cfg.n_heads} / {cfg.n_kv_heads}  (GQA ratio {cfg.n_heads // cfg.n_kv_heads}x)")
print(f"  Encoder layers   : {cfg.num_encoder_layers}")
print(f"  Decoder layers   : {cfg.num_decoder_layers}")
print(f"  FFN d_ff         : {cfg.d_ff}  (SwiGLU)")
print(f"  Vocab size       : {cfg.vocab_size:,}")
print(f"  Max seq length   : {cfg.max_len}")
print(f"  Batch size       : {cfg.batch_size}")
print(f"  Epochs           : {cfg.num_epochs}")
print(f"  LR base          : {cfg.lr_base}")
print(f"  Warmup steps     : {cfg.warmup_steps}")
print(f"  CL epochs        : {cfg.cl_num_epochs}")
print("=" * 55)


  TranslationConfig Summary
  d_model          : 768
  n_heads / n_kv   : 12 / 4  (GQA ratio 3x)
  Encoder layers   : 8
  Decoder layers   : 8
  FFN d_ff         : 3072  (SwiGLU)
  Vocab size       : 8,000
  Max seq length   : 32
  Batch size       : 128
  Epochs           : 40
  LR base          : 0.0002
  Warmup steps     : 200
  CL epochs        : 20


## Cell 4 — Model Architecture
Building blocks of the Transformer:
- **RMSNorm** – faster normalisation without mean subtraction  
- **RoPE** – position encoded via rotation of Q/K vectors  
- **FFN_SwiGLU** – gated feed-forward with Swish × linear gate  
- **GroupedQueryAttentionRoPE** – GQA reduces KV-head memory by `n_heads / n_kv_heads`  
- **EncoderLayer / DecoderLayer** – pre-norm residual blocks  
- **TransformerModel** – full encoder-decoder with tied embeddings  
- **ProjectionHead** – MLP head for contrastive embedding space  


In [5]:
# =============================================================================
# SECTION 2 — MODEL ARCHITECTURE
# =============================================================================

class RMSNorm(nn.Module):
    """Root-Mean-Square Layer Normalisation (faster than LayerNorm)."""

    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.weight * x / rms


class RoPE(nn.Module):
    """Rotary Position Embeddings — encodes position via rotation of Q/K."""

    def __init__(self, d_head: int, base: float = 10_000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer("inv_freq", inv_freq)
        self._seq_len_cached = 0
        self._cos: Optional[torch.Tensor] = None
        self._sin: Optional[torch.Tensor] = None

    def _refresh(self, seq_len: int, device: torch.device, dtype: torch.dtype):
        need_refresh = (
            self._cos is None
            or seq_len > self._seq_len_cached
            or self._cos.device != device
            or self._cos.dtype != dtype
        )
        if need_refresh:
            self._seq_len_cached = seq_len
            pos = torch.arange(seq_len, device=device, dtype=dtype)
            freqs = torch.outer(pos, self.inv_freq.to(device))
            self._cos = freqs.cos()
            self._sin = freqs.sin()

    def forward(
        self, x: torch.Tensor, seq_len: Optional[int] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = seq_len or x.size(-2)
        self._refresh(seq_len, x.device, x.dtype)
        return self._cos[:seq_len], self._sin[:seq_len]  # type: ignore[index]


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to tensor of shape (B, H, T, d_k)."""
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class FFN_SwiGLU(nn.Module):
    """Feed-Forward Network with SwiGLU activation."""

    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.d_ff = d_ff
        self.gate_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj  = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.gate_proj(x)
        g, v = h[..., : self.d_ff], h[..., self.d_ff :]
        return self.dropout(self.out_proj(g * torch.sigmoid(g) * v))


class GroupedQueryAttentionRoPE(nn.Module):
    """
    Grouped-Query Attention (GQA) with Rotary Position Embeddings.
    n_heads Q heads share n_kv_heads K/V heads (n_heads % n_kv_heads == 0).
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: int,
        dropout: float,
        rope_base: float,
    ):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_k        = d_model // n_heads
        self.scale      = math.sqrt(self.d_k)

        self.W_q = nn.Linear(d_model, n_heads    * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model,                bias=False)

        self.dropout = nn.Dropout(dropout)
        self.rope    = RoPE(self.d_k, base=rope_base)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        attn_mask:        Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T_q = q.size(0), q.size(1)
        T_k = k.size(1)

        Q = self.W_q(q).view(B, T_q, self.n_heads,    self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)

        cos_q, sin_q = self.rope(Q, T_q)
        cos_k, sin_k = self.rope(K, T_k)
        Q = apply_rope(Q, cos_q, sin_q)
        K = apply_rope(K, cos_k, sin_k)

        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float("-inf")
            )
        if attn_mask is not None:
            scores = scores.masked_fill(
                attn_mask.unsqueeze(0).unsqueeze(0), float("-inf")
            )

        attn = self.dropout(F.softmax(scores, dim=-1))
        out  = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_q, -1)
        return self.W_o(out)


class EncoderLayer(nn.Module):
    def __init__(self, cfg: TranslationConfig):
        super().__init__()
        self.ln1       = RMSNorm(cfg.d_model)
        self.self_attn = GroupedQueryAttentionRoPE(
            cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.dropout, cfg.rope_base
        )
        self.ln2     = RMSNorm(cfg.d_model)
        self.ffn     = FFN_SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x1 = self.ln1(x)
        x  = x + self.dropout(self.self_attn(x1, x1, x1, key_padding_mask=src_pad_mask))
        return x + self.ffn(self.ln2(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg: TranslationConfig):
        super().__init__()
        self.ln1        = RMSNorm(cfg.d_model)
        self.self_attn  = GroupedQueryAttentionRoPE(
            cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.dropout, cfg.rope_base
        )
        self.ln2        = RMSNorm(cfg.d_model)
        self.cross_attn = GroupedQueryAttentionRoPE(
            cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.dropout, cfg.rope_base
        )
        self.ln3     = RMSNorm(cfg.d_model)
        self.ffn     = FFN_SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(
        self,
        y:               torch.Tensor,
        enc_out:         torch.Tensor,
        tgt_pad_mask:    Optional[torch.Tensor] = None,
        tgt_causal_mask: Optional[torch.Tensor] = None,
        src_pad_mask:    Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        y1 = self.ln1(y)
        y  = y + self.dropout(
            self.self_attn(y1, y1, y1, key_padding_mask=tgt_pad_mask, attn_mask=tgt_causal_mask)
        )
        y2 = self.ln2(y)
        y  = y + self.dropout(self.cross_attn(y2, enc_out, enc_out, key_padding_mask=src_pad_mask))
        return y + self.ffn(self.ln3(y))


class TransformerModel(nn.Module):
    """
    Full Encoder-Decoder Transformer for ZH↔VI translation.
    Architecture highlights:
      · Shared BPE embedding (source + target)
      · Tied output projection weights
      · GQA + RoPE in every attention layer
      · SwiGLU feed-forward
    """

    def __init__(self, cfg: TranslationConfig, vocab_size: int):
        super().__init__()
        self.cfg       = cfg
        self.emb_scale = math.sqrt(cfg.d_model)

        self.embedding     = nn.Embedding(vocab_size, cfg.d_model, padding_idx=0)
        self.emb_dropout   = nn.Dropout(cfg.dropout)

        self.encoder_layers   = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.num_encoder_layers)])
        self.encoder_final_ln = RMSNorm(cfg.d_model)

        self.decoder_layers   = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_decoder_layers)])
        self.decoder_final_ln = RMSNorm(cfg.d_model)

        self.output_bias = nn.Parameter(torch.zeros(vocab_size))
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src_ids: torch.Tensor, src_pad_mask: torch.Tensor) -> torch.Tensor:
        x = self.emb_dropout(self.embedding(src_ids) * self.emb_scale)
        for layer in self.encoder_layers:
            x = layer(x, src_pad_mask)
        return self.encoder_final_ln(x)

    def decode(
        self,
        tgt_ids:         torch.Tensor,
        enc_out:         torch.Tensor,
        tgt_pad_mask:    torch.Tensor,
        tgt_causal_mask: torch.Tensor,
        src_pad_mask:    torch.Tensor,
    ) -> torch.Tensor:
        x = self.emb_dropout(self.embedding(tgt_ids) * self.emb_scale)
        for layer in self.decoder_layers:
            x = layer(x, enc_out, tgt_pad_mask, tgt_causal_mask, src_pad_mask)
        return self.decoder_final_ln(x)

    def project(self, hidden: torch.Tensor) -> torch.Tensor:
        return F.linear(hidden, self.embedding.weight, self.output_bias)

    def forward(self, src_ids: torch.Tensor, tgt_ids: torch.Tensor) -> torch.Tensor:
        src_pad = src_ids == 0
        tgt_pad = tgt_ids == 0

        tgt_in      = tgt_ids[:, :-1]
        tgt_pad_in  = tgt_pad[:, :-1]
        T           = tgt_in.size(1)
        tgt_causal  = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=src_ids.device), diagonal=1
        )

        enc_out = self.encode(src_ids, src_pad)
        dec_out = self.decode(tgt_in, enc_out, tgt_pad_in, tgt_causal, src_pad)
        return self.project(dec_out)


# ── Contrastive projection head ───────────────────────────────────────────────
class ProjectionHead(nn.Module):
    """Maps encoder pooled output to normalised contrastive embedding space."""

    def __init__(self, input_dim: int, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, proj_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), dim=-1)

print("✅ All model classes defined.")


✅ All model classes defined.


## Cell 5 — Model Parameter Count & Architecture Summary
Instantiate the model with a dummy vocab and display a detailed parameter breakdown.

In [6]:
# ── Instantiate model with a dummy vocab size for parameter inspection ────────
_DUMMY_VOCAB = cfg.vocab_size
model_preview = TransformerModel(cfg, _DUMMY_VOCAB).to(cfg.device)

total_params     = sum(p.numel() for p in model_preview.parameters())
trainable_params = sum(p.numel() for p in model_preview.parameters() if p.requires_grad)

print("=" * 60)
print("  Model Architecture Summary")
print("=" * 60)
print(f"  Embedding        : {cfg.vocab_size:,} × {cfg.d_model}  = {cfg.vocab_size * cfg.d_model:,} params")
print(f"  Encoder layers   : {cfg.num_encoder_layers}")
print(f"  Decoder layers   : {cfg.num_decoder_layers}")
print(f"  GQA heads        : {cfg.n_heads} Q-heads / {cfg.n_kv_heads} KV-heads  (group ratio {cfg.n_heads // cfg.n_kv_heads}x)")
print(f"  d_k per head     : {cfg.d_model // cfg.n_heads}")
print(f"  FFN dimension    : {cfg.d_ff}  (SwiGLU gate: 2×{cfg.d_ff}={2*cfg.d_ff})")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print("=" * 60)

# ── Bar chart: parameter distribution per module group ────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

groups = {}
for name, param in model_preview.named_parameters():
    top = name.split(".")[0]
    groups[top] = groups.get(top, 0) + param.numel()

labels = list(groups.keys())
values = [v / 1e6 for v in groups.values()]          # in millions

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels, values, color=plt.cm.tab10.colors[:len(labels)])
ax.set_xlabel("Parameters (millions)")
ax.set_title("Parameter Distribution by Module Group", fontweight="bold")
for bar, val in zip(bars, values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}M", va="center", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.plot_dir, "param_distribution.png"), dpi=130, bbox_inches="tight")
plt.show()
print(f"\n[PLOT] Parameter distribution chart saved.")

del model_preview   # free memory before actual training


  Model Architecture Summary
  Embedding        : 8,000 × 768  = 6,144,000 params
  Encoder layers   : 8
  Decoder layers   : 8
  GQA heads        : 12 Q-heads / 4 KV-heads  (group ratio 3x)
  d_k per head     : 64
  FFN dimension    : 3072  (SwiGLU gate: 2×3072=6144)
  Total params     : 157,179,200
  Trainable params : 157,179,200

[PLOT] Parameter distribution chart saved.


## Cell 6 — Loss Functions & Learning-Rate Scheduler

In [7]:
# =============================================================================
# SECTION 3 — LOSSES & SCHEDULER
# =============================================================================

class LabelSmoothedCrossEntropyLoss(nn.Module):
    def __init__(self, smoothing: float = 0.1, ignore_index: int = 0):
        super().__init__()
        self.smoothing    = smoothing
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        V          = logits.size(-1)
        log_probs  = F.log_softmax(logits, dim=-1)
        mask       = targets != self.ignore_index

        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (V - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            true_dist[targets == self.ignore_index] = 0.0

        loss = -(true_dist * log_probs).sum(dim=-1)
        loss = loss.masked_fill(~mask, 0.0)
        return loss.sum() / mask.sum().clamp(min=1)


class WarmupInverseSqrtScheduler:
    """Linear warm-up then inverse-square-root decay."""

    def __init__(self, optimizer, warmup_steps: int, lr_base: float):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.lr_base      = lr_base
        self.step_num     = 0

    def step(self):
        self.step_num += 1
        s = self.step_num
        if s <= self.warmup_steps:
            lr = self.lr_base * s / self.warmup_steps
        else:
            lr = self.lr_base * math.sqrt(self.warmup_steps / s)
        for g in self.optimizer.param_groups:
            g["lr"] = lr

    def get_lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]


# ── Visualise the LR schedule for 40 epochs (estimated steps) ─────────────────
_steps_per_epoch = 300          # approximate; adjust if known
_total_steps     = cfg.num_epochs * _steps_per_epoch
_lr_curve        = []
_s               = 0
for _i in range(1, _total_steps + 1):
    if _i <= cfg.warmup_steps:
        _lr_curve.append(cfg.lr_base * _i / cfg.warmup_steps)
    else:
        _lr_curve.append(cfg.lr_base * math.sqrt(cfg.warmup_steps / _i))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(_lr_curve, color="steelblue", linewidth=1.2)
ax.axvline(cfg.warmup_steps, color="red", linestyle="--", alpha=0.7,
           label=f"Warmup end (step {cfg.warmup_steps})")
ax.set_title("Warmup + Inverse-Sqrt LR Schedule", fontweight="bold")
ax.set_xlabel("Optimiser step")
ax.set_ylabel("Learning rate")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.plot_dir, "lr_schedule_preview.png"), dpi=130, bbox_inches="tight")
plt.show()
print("✅ LabelSmoothedCrossEntropyLoss and WarmupInverseSqrtScheduler defined.")
print(f"   Label smoothing  : {cfg.label_smoothing}")
print(f"   Warmup steps     : {cfg.warmup_steps}")
print(f"   Peak LR          : {cfg.lr_base}")


✅ LabelSmoothedCrossEntropyLoss and WarmupInverseSqrtScheduler defined.
   Label smoothing  : 0.01
   Warmup steps     : 200
   Peak LR          : 0.0002


## Cell 7 — Dataset & Data Loaders

In [8]:
# =============================================================================
# SECTION 4 — DATASET & DATA LOADING
# =============================================================================

def read_lines(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as fh:
        return [ln.strip() for ln in fh if ln.strip()]


class BidirectionalTranslationDataset(Dataset):
    """
    Bidirectional ZH↔VI dataset with:
    · Language-direction token prefix on source
    · BOS / EOS on target
    · Optional span masking during training
    """

    def __init__(
        self,
        src_lines:   List[str],
        tgt_lines:   List[str],
        sp:          spm.SentencePieceProcessor,
        cfg:         TranslationConfig,
        is_training: bool = True,
    ):
        self.sp          = sp
        self.cfg         = cfg
        self.is_training = is_training

        self.pad_id = sp.piece_to_id(cfg.pad_token)
        self.bos_id = sp.piece_to_id(cfg.bos_token)
        self.eos_id = sp.piece_to_id(cfg.eos_token)
        self.unk_id = sp.piece_to_id(cfg.unk_token)
        self.zh_id  = sp.piece_to_id(cfg.zh_token)
        self.vi_id  = sp.piece_to_id(cfg.vi_token)

        self.samples: List[Tuple[str, str, str]] = []
        if is_training:
            for i, (src, tgt) in enumerate(zip(src_lines, tgt_lines)):
                direction = cfg.vi_token if i % 2 == 0 else cfg.zh_token
                self.samples.append((self._prefix(src, direction), tgt, "zh2vi" if i % 2 == 0 else "vi2zh"))
        else:
            for src, tgt in zip(src_lines, tgt_lines):
                self.samples.append((self._prefix(src, cfg.vi_token), tgt, "zh2vi"))

        self.zh2vi_indices = [i for i, (_, _, d) in enumerate(self.samples) if d == "zh2vi"]
        self.vi2zh_indices = [i for i, (_, _, d) in enumerate(self.samples) if d == "vi2zh"]

    @staticmethod
    def _prefix(text: str, tok: str) -> str:
        text = text.strip()
        return text if text.startswith(("<2vi>", "<2zh>")) else f"{tok} {text}"

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        src_text, tgt_text, _ = self.samples[idx]
        src_ids = self.sp.encode(src_text, out_type=int)[: self.cfg.max_len]
        tgt_ids = ([self.bos_id] + self.sp.encode(tgt_text, out_type=int) + [self.eos_id])[: self.cfg.max_len]
        src_ids = self._span_mask(src_ids)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

    def _span_mask(self, ids: List[int]) -> List[int]:
        if not self.is_training or random.random() > self.cfg.span_mask_prob:
            return ids
        special    = {self.pad_id, self.bos_id, self.eos_id, self.zh_id, self.vi_id}
        maskable   = [i for i, t in enumerate(ids) if t not in special]
        if not maskable:
            return ids
        ids        = ids.copy()
        n          = min(random.randint(1, 2), len(maskable))
        start      = random.choice(maskable)
        for i in range(start, min(start + n, len(ids))):
            if i in maskable and random.random() < 0.7:
                ids[i] = self.unk_id
        return ids


def collate_fn(batch: List[Tuple[torch.Tensor, torch.Tensor]]):
    src_list, tgt_list = zip(*batch)
    max_src = max(s.size(0) for s in src_list)
    max_tgt = max(t.size(0) for t in tgt_list)
    src_pad = torch.zeros(len(batch), max_src, dtype=torch.long)
    tgt_pad = torch.zeros(len(batch), max_tgt, dtype=torch.long)
    for i, (s, t) in enumerate(zip(src_list, tgt_list)):
        src_pad[i, : s.size(0)] = s
        tgt_pad[i, : t.size(0)] = t
    return src_pad, tgt_pad


def _vi2zh_window(indices: List[int], epoch: int, ratio: float) -> List[int]:
    if not indices or ratio <= 0:
        return []
    total  = len(indices)
    window = max(1, int(math.ceil(total * min(ratio, 1.0))))
    start  = ((epoch - 1) * window) % total
    end    = start + window
    return indices[start:end] if end <= total else indices[start:] + indices[:end - total]


def build_train_loader(dataset: BidirectionalTranslationDataset, epoch: int, cfg: TranslationConfig):
    active  = list(dataset.zh2vi_indices)
    active += _vi2zh_window(dataset.vi2zh_indices, epoch, cfg.vi2zh_epoch_ratio)
    sampler = SubsetRandomSampler(active)
    loader  = DataLoader(
        dataset, batch_size=cfg.batch_size, sampler=sampler,
        collate_fn=collate_fn, num_workers=cfg.num_workers, pin_memory=True,
    )
    return loader

print("✅ Dataset and DataLoader utilities defined.")


✅ Dataset and DataLoader utilities defined.


## Cell 8 — Decoding: Greedy & Beam Search

In [9]:
# =============================================================================
# SECTION 5 — DECODING (GREEDY + BEAM SEARCH)
# =============================================================================

@dataclass
class BeamHypothesis:
    tokens:   List[int]
    log_prob: float

    def __lt__(self, other: "BeamHypothesis") -> bool:
        return self.log_prob < other.log_prob


@torch.no_grad()
def greedy_decode(
    model:    TransformerModel,
    src_ids:  torch.Tensor,
    sp:       spm.SentencePieceProcessor,
    cfg:      TranslationConfig,
    max_len:  int = 32,
) -> List[str]:
    model.eval()
    B      = src_ids.size(0)
    device = src_ids.device
    bos    = sp.piece_to_id(cfg.bos_token)
    eos    = sp.piece_to_id(cfg.eos_token)
    pad    = sp.piece_to_id(cfg.pad_token)

    src_pad_mask = src_ids == pad
    enc_out      = model.encode(src_ids, src_pad_mask)
    tgt          = torch.full((B, 1), bos, dtype=torch.long, device=device)

    for _ in range(max_len):
        T           = tgt.size(1)
        causal      = torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)
        dec_out     = model.decode(tgt, enc_out, tgt == pad, causal, src_pad_mask)
        next_tok    = model.project(dec_out)[:, -1, :].argmax(dim=-1, keepdim=True)
        tgt         = torch.cat([tgt, next_tok], dim=1)
        if (next_tok.squeeze(-1) == eos).all():
            break

    decoded = []
    for seq in tgt:
        tokens = seq[1:].tolist()
        if eos in tokens:
            tokens = tokens[: tokens.index(eos)]
        decoded.append(sp.decode(tokens))
    return decoded


@torch.no_grad()
def beam_search_decode(
    model:          TransformerModel,
    src_ids:        torch.Tensor,
    sp:             spm.SentencePieceProcessor,
    cfg:            TranslationConfig,
    beam_size:      int   = 3,
    top_k:          int   = 5,
    max_len:        int   = 32,
    length_penalty: float = 0.6,
) -> List[str]:
    model.eval()
    B      = src_ids.size(0)
    device = src_ids.device
    bos    = sp.piece_to_id(cfg.bos_token)
    eos    = sp.piece_to_id(cfg.eos_token)
    pad    = sp.piece_to_id(cfg.pad_token)

    src_pad_mask = src_ids == pad
    enc_out      = model.encode(src_ids, src_pad_mask)

    results = []
    for b in range(B):
        beams:    List[BeamHypothesis] = [BeamHypothesis(tokens=[bos], log_prob=0.0)]
        finished: List[BeamHypothesis] = []
        enc_b    = enc_out[b : b + 1]
        mask_b   = src_pad_mask[b : b + 1]

        for _ in range(max_len):
            proposals: List[BeamHypothesis] = []
            for hyp in beams:
                if hyp.tokens[-1] == eos:
                    finished.append(hyp)
                    continue
                tgt_t   = torch.tensor(hyp.tokens, dtype=torch.long, device=device).unsqueeze(0)
                T       = tgt_t.size(1)
                causal  = torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)
                dec_out = model.decode(tgt_t, enc_b, tgt_t == pad, causal, mask_b)
                logits  = model.project(dec_out)
                lp      = F.log_softmax(logits[:, -1, :], dim=-1)

                if top_k and top_k > 0:
                    tv, ti  = torch.topk(lp, min(top_k, lp.size(-1)))
                    filtered = torch.full_like(lp, float("-inf"))
                    filtered.scatter_(1, ti, tv)
                    lp = filtered

                top_v, top_i = torch.topk(lp.squeeze(0), beam_size)
                for v, i in zip(top_v.tolist(), top_i.tolist()):
                    proposals.append(BeamHypothesis(tokens=hyp.tokens + [i], log_prob=hyp.log_prob + v))

            beams = sorted(proposals, key=lambda h: h.log_prob, reverse=True)[:beam_size]
            if not beams:
                break

        finished.extend(beams)
        best = max(finished, key=lambda h: h.log_prob / (len(h.tokens) ** length_penalty))
        tokens = best.tokens[1:]
        if eos in tokens:
            tokens = tokens[: tokens.index(eos)]
        results.append(sp.decode(tokens))
    return results


print("✅ Greedy decode and beam search decode defined.")
print(f"   Beam size      : {cfg.beam_size}")
print(f"   Top-k filter   : {cfg.top_k}")
print(f"   Length penalty : {cfg.length_penalty}")


✅ Greedy decode and beam search decode defined.
   Beam size      : 3
   Top-k filter   : 5
   Length penalty : 0.6


## Cell 9 — Evaluation (Loss + BLEU)

In [10]:
# =============================================================================
# SECTION 6 — EVALUATION
# =============================================================================

@torch.no_grad()
def evaluate(
    model:            TransformerModel,
    dataloader:       DataLoader,
    criterion:        LabelSmoothedCrossEntropyLoss,
    sp:               spm.SentencePieceProcessor,
    cfg:              TranslationConfig,
    calculate_bleu:   bool = True,
    max_bleu_samples: int  = 300,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    preds, refs = [], []
    bleu_count  = 0

    eos = sp.piece_to_id(cfg.eos_token)

    for src_b, tgt_b in tqdm(dataloader, desc="Evaluating", leave=False):
        src_b = src_b.to(cfg.device)
        tgt_b = tgt_b.to(cfg.device)

        logits  = model(src_b, tgt_b)
        targets = tgt_b[:, 1:]
        loss    = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        total_loss += loss.item()

        if calculate_bleu and bleu_count < max_bleu_samples:
            batch_preds = greedy_decode(model, src_b, sp, cfg, max_len=cfg.max_len)
            for tgt_seq in tgt_b:
                ref = tgt_seq[1:].cpu().tolist()
                if eos in ref:
                    ref = ref[: ref.index(eos)]
                refs.append(sp.decode(ref))
            preds.extend(batch_preds)
            bleu_count += len(batch_preds)

    avg_loss   = total_loss / max(1, len(dataloader))
    bleu_score = 0.0

    if calculate_bleu and SACREBLEU_AVAILABLE and preds:
        try:
            bleu_score = sacrebleu_lib.corpus_bleu(preds, [refs], force=True).score
        except Exception as e:
            print(f"[WARN] BLEU calculation failed: {e}")

    return avg_loss, bleu_score

print("✅ Evaluation function defined.")
print(f"   Max BLEU samples per evaluation : {cfg.max_bleu_samples}")
print(f"   SacreBLEU available             : {SACREBLEU_AVAILABLE}")


✅ Evaluation function defined.
   Max BLEU samples per evaluation : 300
   SacreBLEU available             : True


## Cell 10 — Visualisation Helpers
Functions to plot: token length distributions, training curves, contrastive curves.

In [11]:
# =============================================================================
# SECTION 7 — VISUALISATION HELPERS
# =============================================================================

def plot_training_curves(
    train_losses: List[float],
    val_losses:   List[float],
    val_bleus:    List[float],
    lr_history:   List[float],
    save_dir:     str,
) -> None:
    """Save an animated-style training dashboard (4-panel figure)."""
    epochs = list(range(1, len(train_losses) + 1))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("ZH-VI Transformer — Training Dashboard", fontsize=15, fontweight="bold")

    # ── Panel 1: Train vs Val Loss ────────────────────────────────────────────
    ax = axes[0, 0]
    ax.plot(epochs, train_losses, "b-o", markersize=4, label="Train Loss")
    ax.plot(epochs, val_losses,   "r-s", markersize=4, label="Val Loss")
    best_epoch = int(np.argmin(val_losses)) + 1
    best_loss  = min(val_losses)
    ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best @ ep{best_epoch}")
    ax.annotate(
        f"  {best_loss:.4f}",
        xy=(best_epoch, best_loss),
        fontsize=8, color="green",
    )
    ax.set_title("Loss Curves")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)

    # ── Panel 2: BLEU Score ───────────────────────────────────────────────────
    ax = axes[0, 1]
    ax.fill_between(epochs, val_bleus, alpha=0.2, color="orange")
    ax.plot(epochs, val_bleus, "orange", marker="D", markersize=4, label="Val BLEU")
    best_bleu_ep = int(np.argmax(val_bleus)) + 1
    ax.axvline(best_bleu_ep, color="purple", linestyle="--", alpha=0.6, label=f"Best @ ep{best_bleu_ep}")
    ax.set_title("Validation BLEU Score")
    ax.set_xlabel("Epoch"); ax.set_ylabel("BLEU")
    ax.legend(); ax.grid(True, alpha=0.3)

    # ── Panel 3: Train loss (log scale) ──────────────────────────────────────
    ax = axes[1, 0]
    ax.semilogy(epochs, train_losses, "b-o", markersize=4)
    ax.set_title("Train Loss (log scale)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log)")
    ax.grid(True, alpha=0.3, which="both")

    # ── Panel 4: Learning-rate schedule ──────────────────────────────────────
    ax = axes[1, 1]
    ax.plot(lr_history, "g-", linewidth=0.8, alpha=0.8)
    ax.set_title("Learning Rate Schedule (per step)")
    ax.set_xlabel("Optimiser step"); ax.set_ylabel("LR")
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Saved training curves → {path}")


def plot_contrastive_curves(
    ce_losses:    List[float],
    cl_losses:    List[float],
    total_losses: List[float],
    save_dir:     str,
) -> None:
    epochs = list(range(1, len(ce_losses) + 1))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Contrastive Fine-Tuning Losses", fontsize=13, fontweight="bold")

    titles   = ["Cross-Entropy Loss", "Contrastive Loss", "Total Loss"]
    data     = [ce_losses, cl_losses, total_losses]
    colours  = ["steelblue", "tomato", "mediumpurple"]

    for ax, title, vals, colour in zip(axes, titles, data, colours):
        ax.fill_between(epochs, vals, alpha=0.15, color=colour)
        ax.plot(epochs, vals, color=colour, marker="o", markersize=4)
        ax.set_title(title)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(save_dir, "contrastive_curves.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Saved contrastive curves → {path}")


def plot_token_length_distribution(
    zh_before: np.ndarray,
    vi_before: np.ndarray,
    zh_after:  np.ndarray,
    vi_after:  np.ndarray,
    max_len:   int,
    save_dir:  str,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Token Length Distribution (Before vs After Filtering)", fontsize=13, fontweight="bold")
    pairs = [
        (axes[0, 0], zh_before, "ZH  — Before filtering", "royalblue"),
        (axes[0, 1], vi_before, "VI  — Before filtering",  "seagreen"),
        (axes[1, 0], zh_after,  f"ZH  — After (≤{max_len} tokens)", "cornflowerblue"),
        (axes[1, 1], vi_after,  f"VI  — After (≤{max_len} tokens)",  "mediumseagreen"),
    ]
    for ax, data, title, colour in pairs:
        ax.hist(data, bins=40, color=colour, alpha=0.75, edgecolor="white")
        ax.axvline(max_len, color="red", linestyle="--", label=f"max_len={max_len}")
        ax.set_title(title)
        ax.set_xlabel("Token count")
        ax.set_ylabel("Sentences")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "length_distribution.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[PLOT] Saved length distribution → {path}")

print("✅ Visualisation helpers defined.")


✅ Visualisation helpers defined.


## Cell 11 — Checkpoint Save / Load

In [12]:
# =============================================================================
# SECTION 8 — CHECKPOINT HELPERS
# =============================================================================

def _tokenizer_payload(prefix: str) -> Dict[str, Any]:
    for ext in (".model", ".vocab"):
        if not os.path.isfile(f"{prefix}{ext}"):
            raise FileNotFoundError(f"Missing tokenizer file: {prefix}{ext}")
    with open(f"{prefix}.model", "rb") as fh:
        model_bytes = fh.read()
    with open(f"{prefix}.vocab", "rb") as fh:
        vocab_bytes = fh.read()
    return {"prefix": prefix, "model_bytes": model_bytes, "vocab_bytes": vocab_bytes}


def save_checkpoint(
    path:       str,
    model:      TransformerModel,
    optimizer,
    scheduler,
    epoch:      int,
    train_loss: float,
    val_loss:   float,
    val_bleu:   float,
    cfg:        TranslationConfig,
    tokenizer_prefix: str,
    extra: Optional[Dict] = None,
) -> None:
    payload = {
        "epoch":               epoch,
        "model_state_dict":    model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss":          train_loss,
        "val_loss":            val_loss,
        "val_bleu":            val_bleu,
        "config":              cfg,
        "tokenizer":           _tokenizer_payload(tokenizer_prefix),
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_model_from_checkpoint(
    checkpoint_path: str,
    device:          torch.device,
) -> Tuple["TransformerModel", "spm.SentencePieceProcessor", "TranslationConfig"]:
    """Load model + tokenizer from a self-contained checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg  = ckpt.get("config", TranslationConfig())
    cfg.device = device

    tok_payload = ckpt.get("tokenizer")
    if not tok_payload:
        raise ValueError("Checkpoint does not contain embedded tokenizer.")

    tmp_dir = tempfile.mkdtemp(prefix="spm_ckpt_")
    prefix  = os.path.join(tmp_dir, "spm_from_ckpt")
    with open(f"{prefix}.model", "wb") as fh:
        fh.write(tok_payload["model_bytes"])
    with open(f"{prefix}.vocab", "wb") as fh:
        fh.write(tok_payload["vocab_bytes"])

    sp = spm.SentencePieceProcessor()
    sp.Load(f"{prefix}.model")
    vocab_size = sp.GetPieceSize()

    model = TransformerModel(cfg, vocab_size).to(device)
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    model.eval()
    return model, sp, cfg

print("✅ Checkpoint save/load helpers defined.")


✅ Checkpoint save/load helpers defined.


## Cell 12 — Contrastive Learning Helpers

In [13]:
# =============================================================================
# SECTION 9 — CONTRASTIVE HELPERS
# =============================================================================

def _encode_context(
    model: TransformerModel,
    src_ids: torch.Tensor,
    pad_id: int,
) -> torch.Tensor:
    src_pad = src_ids == pad_id
    return model.encode(src_ids, src_pad)


def _mean_pool(
    enc_out:     torch.Tensor,
    ids:         torch.Tensor,
    pad_id:      int,
    special_ids: List[int],
) -> torch.Tensor:
    mask = ids != pad_id
    for sid in special_ids:
        mask = mask & (ids != sid)
    mask   = mask.float()
    summed = (enc_out * mask.unsqueeze(-1)).sum(dim=1)
    denom  = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    return summed / denom


def _contrastive_loss(z_a: torch.Tensor, z_b: torch.Tensor, tau: float) -> torch.Tensor:
    sim    = torch.matmul(z_a, z_b.T) / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))


class ContrastiveBatchDataset(Dataset):
    """Wraps training pairs to supply four-way anchor/positive batches."""

    def __init__(
        self,
        src_lines: List[str],
        tgt_lines: List[str],
        sp:        spm.SentencePieceProcessor,
        cfg:       TranslationConfig,
    ):
        self.base = BidirectionalTranslationDataset(src_lines, tgt_lines, sp, cfg, is_training=True)
        self.sp   = sp
        self.cfg  = cfg

        self.pad_id = sp.piece_to_id(cfg.pad_token)
        self.bos_id = sp.piece_to_id(cfg.bos_token)
        self.eos_id = sp.piece_to_id(cfg.eos_token)
        self.zh_id  = sp.piece_to_id(cfg.zh_token)
        self.vi_id  = sp.piece_to_id(cfg.vi_token)

    def __len__(self) -> int:
        return len(self.base)

    def _encode(self, text: str, prefix: Optional[str] = None) -> List[int]:
        t = f"{prefix} {text}" if prefix else text
        return self.sp.encode(t.strip(), out_type=int)[: self.cfg.max_len]

    def __getitem__(self, idx: int):
        src_text, tgt_text, direction = self.base.samples[idx]
        src_ids, tgt_ids = self.base[idx]

        # Build four anchors for cross-lingual contrastive learning
        ids_zh_vi = torch.tensor(self._encode(src_text, self.cfg.vi_token), dtype=torch.long)
        ids_vi_vi = torch.tensor(self._encode(tgt_text),                     dtype=torch.long)
        ids_vi_zh = torch.tensor(self._encode(tgt_text, self.cfg.zh_token),  dtype=torch.long)
        ids_zh_zh = torch.tensor(self._encode(src_text, self.cfg.zh_token),  dtype=torch.long)

        return {
            "src":      src_ids,
            "tgt":      tgt_ids,
            "ids_zh_vi": ids_zh_vi,
            "ids_vi_vi": ids_vi_vi,
            "ids_vi_zh": ids_vi_zh,
            "ids_zh_zh": ids_zh_zh,
        }


def _cl_collate(batch):
    keys = list(batch[0].keys())
    out  = {}
    for k in keys:
        seqs   = [item[k] for item in batch]
        max_l  = max(s.size(0) for s in seqs)
        padded = torch.zeros(len(batch), max_l, dtype=torch.long)
        for i, s in enumerate(seqs):
            padded[i, : s.size(0)] = s
        out[k] = padded
    return out

print("✅ Contrastive learning helpers defined.")
print(f"   Temperature τ    : {cfg.cl_tau}")
print(f"   Lambda max       : {cfg.cl_lambda_max}")
print(f"   CL warmup steps  : {cfg.cl_warmup_steps}")


✅ Contrastive learning helpers defined.
   Temperature τ    : 0.07
   Lambda max       : 0.1
   CL warmup steps  : 200


## Cell 13 — Training Loops (Main + Contrastive)

In [14]:
# =============================================================================
# SECTION 10 — TRAINING LOOPS
# =============================================================================

def train_one_epoch(
    model:     TransformerModel,
    loader:    DataLoader,
    criterion: LabelSmoothedCrossEntropyLoss,
    optimizer,
    scheduler: WarmupInverseSqrtScheduler,
    cfg:       TranslationConfig,
    epoch:     int,
    lr_history: List[float],
) -> float:
    model.train()
    total = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [train]")
    for src_b, tgt_b in pbar:
        src_b = src_b.to(cfg.device)
        tgt_b = tgt_b.to(cfg.device)

        logits = model(src_b, tgt_b)
        loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_b[:, 1:].reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        total += loss.item()
        lr_history.append(scheduler.get_lr())
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_lr():.2e}")

    return total / max(1, len(loader))


def run_training(
    model:      TransformerModel,
    train_ds:   BidirectionalTranslationDataset,
    valid_dl:   DataLoader,
    sp:         spm.SentencePieceProcessor,
    cfg:        TranslationConfig,
    tokenizer_prefix: str,
) -> Tuple[List[float], List[float], List[float], List[float], str]:
    """Full training loop. Returns (train_losses, val_losses, val_bleus, lr_history, best_path)."""
    criterion = LabelSmoothedCrossEntropyLoss(cfg.label_smoothing, ignore_index=0)
    optimizer = optim.AdamW(
        model.parameters(), lr=cfg.lr_base,
        betas=(0.9, 0.98), eps=1e-9, weight_decay=cfg.weight_decay,
    )
    scheduler = WarmupInverseSqrtScheduler(optimizer, cfg.warmup_steps, cfg.lr_base)

    train_losses: List[float] = []
    val_losses:   List[float] = []
    val_bleus:    List[float] = []
    lr_history:   List[float] = []

    best_val_loss = float("inf")
    best_val_bleu = 0.0
    best_path     = os.path.join(cfg.checkpoint_dir, "best_model_zh_vi_transformer.pth")

    for epoch in range(1, cfg.num_epochs + 1):
        train_loader = build_train_loader(train_ds, epoch, cfg)
        train_loss   = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, cfg, epoch, lr_history)
        val_loss, val_bleu = evaluate(model, valid_dl, criterion, sp, cfg,
                                      calculate_bleu=True, max_bleu_samples=cfg.max_bleu_samples)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_bleus.append(val_bleu)

        print(
            f"\n[Epoch {epoch:03d}/{cfg.num_epochs}]  "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  "
            f"val_BLEU={val_bleu:.2f}  "
            f"lr={scheduler.get_lr():.2e}"
        )

        if epoch % cfg.save_every == 0:
            ckpt_path = os.path.join(cfg.checkpoint_dir, f"checkpoint_epoch_{epoch:04d}.pt")
            save_checkpoint(
                ckpt_path, model, optimizer, scheduler,
                epoch, train_loss, val_loss, val_bleu, cfg, tokenizer_prefix,
            )
            print(f"  ✔ Checkpoint saved → {ckpt_path}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_bleu = val_bleu
            save_checkpoint(
                best_path, model, optimizer, scheduler,
                epoch, train_loss, val_loss, val_bleu, cfg, tokenizer_prefix,
            )
            print(f"  ★ New best model saved (val_loss={best_val_loss:.4f}  BLEU={best_val_bleu:.2f})")

    print(f"\n[Training complete]  best_val_loss={best_val_loss:.4f}  best_BLEU={best_val_bleu:.2f}")
    return train_losses, val_losses, val_bleus, lr_history, best_path

print("✅ Training loop functions defined.")


✅ Training loop functions defined.


## Cell 14 — Contrastive Fine-Tuning Loop

In [15]:
# =============================================================================
# SECTION 11 — CONTRASTIVE FINE-TUNING LOOP
# =============================================================================

def contrastive_train_one_epoch(
    model:       TransformerModel,
    projection:  ProjectionHead,
    loader:      DataLoader,
    criterion:   LabelSmoothedCrossEntropyLoss,
    optimizer,
    scheduler:   WarmupInverseSqrtScheduler,
    cfg:         TranslationConfig,
    special_ids: List[int],
    epoch:       int,
    global_step: int,
) -> Tuple[float, float, float, int]:
    model.train()
    projection.train()

    pad_id    = special_ids[0]
    total_ce  = 0.0
    total_cl  = 0.0
    total_tot = 0.0

    pbar = tqdm(loader, desc=f"CL Epoch {epoch:03d}")
    for batch in pbar:
        src       = batch["src"].to(cfg.device)
        tgt       = batch["tgt"].to(cfg.device)
        ids_zh_vi = batch["ids_zh_vi"].to(cfg.device)
        ids_vi_vi = batch["ids_vi_vi"].to(cfg.device)
        ids_vi_zh = batch["ids_vi_zh"].to(cfg.device)
        ids_zh_zh = batch["ids_zh_zh"].to(cfg.device)

        optimizer.zero_grad()

        # Cross-entropy loss
        logits  = model(src, tgt)
        ce_loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))

        # Contrastive loss with linear warmup of lambda
        warmup_ratio = min(1.0, global_step / max(1, cfg.cl_warmup_steps))
        lam          = cfg.cl_lambda_max * warmup_ratio

        if ids_zh_vi.size(0) >= 2:
            z_zh_vi = projection(_mean_pool(_encode_context(model, ids_zh_vi, pad_id), ids_zh_vi, pad_id, special_ids))
            z_vi_vi = projection(_mean_pool(_encode_context(model, ids_vi_vi, pad_id), ids_vi_vi, pad_id, special_ids))
            z_vi_zh = projection(_mean_pool(_encode_context(model, ids_vi_zh, pad_id), ids_vi_zh, pad_id, special_ids))
            z_zh_zh = projection(_mean_pool(_encode_context(model, ids_zh_zh, pad_id), ids_zh_zh, pad_id, special_ids))
            cl_loss = 0.5 * (
                _contrastive_loss(z_zh_vi, z_vi_vi, cfg.cl_tau) +
                _contrastive_loss(z_vi_zh, z_zh_zh, cfg.cl_tau)
            )
        else:
            cl_loss = torch.zeros(1, device=cfg.device).squeeze()

        total_loss = ce_loss + lam * cl_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(projection.parameters()), cfg.grad_clip
        )
        optimizer.step()
        scheduler.step()

        total_ce  += ce_loss.item()
        total_cl  += cl_loss.item()
        total_tot += total_loss.item()
        global_step += 1

        pbar.set_postfix(
            ce=f"{ce_loss.item():.4f}",
            cl=f"{cl_loss.item():.4f}",
            lam=f"{lam:.3f}",
        )

    n = max(1, len(loader))
    return total_ce / n, total_cl / n, total_tot / n, global_step


def run_contrastive_finetuning(
    model:            TransformerModel,
    train_src_lines:  List[str],
    train_tgt_lines:  List[str],
    sp:               spm.SentencePieceProcessor,
    cfg:              TranslationConfig,
    tokenizer_prefix: str,
) -> Tuple[List[float], List[float], List[float]]:
    projection = ProjectionHead(cfg.d_model, cfg.cl_proj_dim).to(cfg.device)

    special_ids = [
        sp.piece_to_id(cfg.pad_token),
        sp.piece_to_id(cfg.bos_token),
        sp.piece_to_id(cfg.eos_token),
        sp.piece_to_id(cfg.zh_token),
        sp.piece_to_id(cfg.vi_token),
    ]

    cl_dataset = ContrastiveBatchDataset(train_src_lines, train_tgt_lines, sp, cfg)
    cl_loader  = DataLoader(
        cl_dataset, batch_size=cfg.cl_batch_size, shuffle=True,
        collate_fn=_cl_collate, num_workers=cfg.num_workers, pin_memory=True,
    )

    optimizer = optim.AdamW(
        list(model.parameters()) + list(projection.parameters()),
        lr=cfg.cl_lr, betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01,
    )
    scheduler = WarmupInverseSqrtScheduler(optimizer, cfg.cl_warmup_steps, cfg.cl_lr)
    criterion = LabelSmoothedCrossEntropyLoss(0.01, ignore_index=0)

    ce_hist:    List[float] = []
    cl_hist:    List[float] = []
    tot_hist:   List[float] = []
    best_loss   = float("inf")
    global_step = 0
    best_path   = os.path.join(cfg.contrastive_dir, "best_contrastive_model_zh_vi.pth")

    for epoch in range(1, cfg.cl_num_epochs + 1):
        ce, cl, tot, global_step = contrastive_train_one_epoch(
            model, projection, cl_loader, criterion, optimizer, scheduler,
            cfg, special_ids, epoch, global_step,
        )
        ce_hist.append(ce)
        cl_hist.append(cl)
        tot_hist.append(tot)

        print(f"\n[CL Epoch {epoch:03d}/{cfg.cl_num_epochs}]  "
              f"CE={ce:.4f}  CL={cl:.4f}  Total={tot:.4f}  lr={scheduler.get_lr():.2e}")

        if epoch % cfg.cl_save_every == 0:
            ckpt_path = os.path.join(cfg.contrastive_dir, f"cl_checkpoint_epoch_{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "projection_state_dict": projection.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "ce_loss": ce, "cl_loss": cl, "total_loss": tot,
                "config": cfg, "tokenizer": _tokenizer_payload(tokenizer_prefix),
            }, ckpt_path)
            print(f"  ✔ CL checkpoint saved → {ckpt_path}")

        if tot < best_loss:
            best_loss = tot
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "projection_state_dict": projection.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "ce_loss": ce, "cl_loss": cl, "total_loss": tot,
                "config": cfg, "tokenizer": _tokenizer_payload(tokenizer_prefix),
            }, best_path)
            print(f"  ★ New best CL model saved (total={best_loss:.4f})")

    print(f"\n[Contrastive fine-tuning complete]  best_total_loss={best_loss:.4f}")
    return ce_hist, cl_hist, tot_hist

print("✅ Contrastive fine-tuning loop defined.")


✅ Contrastive fine-tuning loop defined.


## Cell 15 — Inference & Test Output

In [16]:
# =============================================================================
# SECTION 12 — INFERENCE & TEST OUTPUT
# =============================================================================

def translate_file(
    model:        TransformerModel,
    sp:           spm.SentencePieceProcessor,
    cfg:          TranslationConfig,
    input_path:   str,
    output_path:  str,
    direction_token: str = "<2vi>",
    split_name:   str = "Test",
) -> pd.DataFrame:
    """Translate all sentences in input_path and save to output_path (CSV)."""
    sentences = read_lines(input_path)
    rows      = []
    model.eval()

    for sentence in tqdm(sentences, desc=f"Translating [{split_name}]"):
        src_text   = f"{direction_token} {sentence}".strip()
        src_ids    = sp.encode(src_text, out_type=int)[: cfg.max_len]
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=cfg.device).unsqueeze(0)

        translation = beam_search_decode(
            model, src_tensor, sp, cfg,
            beam_size=cfg.beam_size, top_k=cfg.top_k,
            max_len=cfg.max_len, length_penalty=cfg.length_penalty,
        )[0]
        rows.append({"source_zh": sentence, "translation_vi": translation})

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"[INFERENCE] Saved {len(df)} translations → {output_path}")
    return df


def show_sample_translations(df: pd.DataFrame, n: int = 10) -> None:
    print(f"\n{'='*70}")
    print(f"  SAMPLE TEST TRANSLATIONS (first {n} rows)")
    print(f"{'='*70}")
    sample = df.head(n)
    for i, row in sample.iterrows():
        print(f"\n  [{i+1:02d}] ZH : {row['source_zh']}")
        print(f"       VI : {row['translation_vi']}")
    print(f"\n{'='*70}")

print("✅ Inference helpers defined.")


✅ Inference helpers defined.


## Cell 16 — Main Pipeline (Steps 1–9)
Runs the full end-to-end pipeline:
1. Load raw data
2. Train BPE tokenizer
3. Filter by token length + plot distributions
4. Build datasets & loaders
5. Build model
6. Train (40 epochs)
7. Plot training curves
8. Contrastive fine-tune (20 epochs)
9. Inference on public/private test sets


In [17]:
# =============================================================================
# SECTION 13 — MAIN PIPELINE
# =============================================================================

def main():
    print("=" * 70)
    print("  ZH-VI Transformer — Full Pipeline")
    print(f"  Device : {cfg.device}")
    print("=" * 70)

    # ── Step 1: Load raw data ──────────────────────────────────────────────────
    print("\n[STEP 1] Loading raw data ...")
    zh_lines = read_lines(cfg.train_zh_file)
    vi_lines = read_lines(cfg.train_vi_file)
    assert len(zh_lines) == len(vi_lines), "Mismatch in source/target line counts."
    print(f"  Loaded {len(zh_lines):,} sentence pairs.")

    zh_raw_len = np.array([len(s.split()) for s in zh_lines])
    vi_raw_len = np.array([len(s.split()) for s in vi_lines])
    print(f"  ZH  word-length: mean={zh_raw_len.mean():.1f}  max={zh_raw_len.max()}")
    print(f"  VI  word-length: mean={vi_raw_len.mean():.1f}  max={vi_raw_len.max()}")

    # ── Step 2: Train tokenizer ────────────────────────────────────────────────
    print("\n[STEP 2] Training BPE tokenizer ...")
    tmp_corpus = os.path.join(cfg.tokenizer_dir, "tmp_corpus.txt")
    with open(tmp_corpus, "w", encoding="utf-8") as fh:
        for z, v in zip(zh_lines, vi_lines):
            fh.write(z + "\n")
            fh.write(v + "\n")

    spm.SentencePieceTrainer.Train(
        f"--input={tmp_corpus} "
        f"--model_prefix={TOKENIZER_PREFIX} "
        f"--vocab_size={cfg.vocab_size} "
        f"--model_type=bpe "
        f"--character_coverage=1.0 "
        f"--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3 "
        f"--user_defined_symbols=<2zh>,<2vi>"
    )
    print(f"  Tokenizer saved → {TOKENIZER_PREFIX}.model")

    sp = spm.SentencePieceProcessor()
    sp.Load(f"{TOKENIZER_PREFIX}.model")
    VOCAB_SIZE = sp.GetPieceSize()
    print(f"  Vocabulary size: {VOCAB_SIZE:,}")

    # ── Step 3: Filter by token length ────────────────────────────────────────
    print(f"\n[STEP 3] Filtering sentences (max {cfg.max_len} tokens) ...")

    def tok_len(t: str, prefix: Optional[str] = None) -> int:
        t = f"{prefix} {t}" if prefix else t
        return len(sp.encode(t, out_type=int))

    zh_before = np.array([tok_len(z, "<2vi>") for z in zh_lines])
    vi_before = np.array([tok_len(v)           for v in vi_lines])

    filt_zh, filt_vi = zip(*(
        (z, v) for z, v in zip(zh_lines, vi_lines)
        if tok_len(z, "<2vi>") <= cfg.max_len and tok_len(v) <= cfg.max_len
    ))
    filt_zh, filt_vi = list(filt_zh), list(filt_vi)

    zh_after = np.array([tok_len(z, "<2vi>") for z in filt_zh])
    vi_after = np.array([tok_len(v)           for v in filt_vi])

    print(f"  Kept {len(filt_zh):,} / {len(zh_lines):,} pairs.")

    zh_out = os.path.join(cfg.clean_data_dir, f"train_maxlen{cfg.max_len}.zh")
    vi_out = os.path.join(cfg.clean_data_dir, f"train_maxlen{cfg.max_len}.vi")
    with open(zh_out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(filt_zh))
    with open(vi_out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(filt_vi))

    # Plot length distributions (inline display)
    plot_token_length_distribution(zh_before, vi_before, zh_after, vi_after, cfg.max_len, cfg.plot_dir)

    # ── Step 4: Build datasets & loaders ──────────────────────────────────────
    print("\n[STEP 4] Building datasets ...")
    split_idx = int((1 - cfg.val_split) * len(filt_zh))
    split_idx -= split_idx % 2

    train_zh, train_vi = filt_zh[:split_idx], filt_vi[:split_idx]
    valid_zh, valid_vi = filt_zh[split_idx:], filt_vi[split_idx:]
    valid_zh = valid_zh[0::2]
    valid_vi = valid_vi[0::2]

    train_ds = BidirectionalTranslationDataset(train_zh, train_vi, sp, cfg, is_training=True)
    valid_ds = BidirectionalTranslationDataset(valid_zh, valid_vi, sp, cfg, is_training=False)
    valid_dl = DataLoader(
        valid_ds, batch_size=cfg.batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=cfg.num_workers, pin_memory=True,
    )

    print(f"  Train pairs: {len(train_ds):,}   "
          f"(zh→vi: {len(train_ds.zh2vi_indices):,}  vi→zh: {len(train_ds.vi2zh_indices):,})")
    print(f"  Valid pairs: {len(valid_ds):,}")

    # ── Step 5: Build model ────────────────────────────────────────────────────
    print("\n[STEP 5] Building model ...")
    model = TransformerModel(cfg, VOCAB_SIZE).to(cfg.device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {total_params:,}")
    print(f"  Architecture: {cfg.num_encoder_layers} encoder / {cfg.num_decoder_layers} decoder layers")
    print(f"  Attention: GQA ({cfg.n_heads} Q-heads / {cfg.n_kv_heads} KV-heads) + RoPE")
    print(f"  FFN: SwiGLU  ({cfg.d_model} → {2*cfg.d_ff} → {cfg.d_ff} → {cfg.d_model})")

    # ── Step 6: Training ──────────────────────────────────────────────────────
    print(f"\n[STEP 6] Training for {cfg.num_epochs} epochs ...")
    train_losses, val_losses, val_bleus, lr_history, best_path = run_training(
        model, train_ds, valid_dl, sp, cfg, TOKENIZER_PREFIX
    )

    # ── Step 7: Plot training curves (inline display) ─────────────────────────
    print("\n[STEP 7] Plotting training curves ...")
    plot_training_curves(train_losses, val_losses, val_bleus, lr_history, cfg.plot_dir)

    # ── Step 7b: Metrics summary table ────────────────────────────────────────
    epochs_range = list(range(1, len(train_losses) + 1))
    metrics_df = pd.DataFrame({
        "Epoch":      epochs_range,
        "Train Loss": [f"{v:.4f}" for v in train_losses],
        "Val Loss":   [f"{v:.4f}" for v in val_losses],
        "Val BLEU":   [f"{v:.2f}" for v in val_bleus],
    })
    best_ep = int(np.argmin(val_losses))
    print("\n  ─── Best epoch metrics ───────────────────────")
    print(f"  Best epoch       : {best_ep + 1}")
    print(f"  Best val loss    : {val_losses[best_ep]:.4f}")
    print(f"  Best val BLEU    : {val_bleus[best_ep]:.2f}")
    print(f"  Final train loss : {train_losses[-1]:.4f}")
    print(f"  Peak BLEU        : {max(val_bleus):.2f}  @ epoch {int(np.argmax(val_bleus)) + 1}")
    print("  ──────────────────────────────────────────────")

    # Interactive metrics table (last 10 epochs)
    from IPython.display import display
    print("\n  Last 10 epochs:")
    display(metrics_df.tail(10).reset_index(drop=True))

    # ── Step 8: Contrastive fine-tuning ───────────────────────────────────────
    print(f"\n[STEP 8] Contrastive fine-tuning for {cfg.cl_num_epochs} epochs ...")
    best_ckpt = torch.load(best_path, map_location=cfg.device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])
    print(f"  Loaded best model from epoch {best_ckpt['epoch']} "
          f"(val_loss={best_ckpt['val_loss']:.4f}  BLEU={best_ckpt['val_bleu']:.2f})")

    ce_hist, cl_hist, tot_hist = run_contrastive_finetuning(
        model, train_zh, train_vi, sp, cfg, TOKENIZER_PREFIX
    )
    plot_contrastive_curves(ce_hist, cl_hist, tot_hist, cfg.plot_dir)

    # Contrastive metrics summary
    cl_df = pd.DataFrame({
        "Epoch":  list(range(1, len(ce_hist) + 1)),
        "CE Loss":    [f"{v:.4f}" for v in ce_hist],
        "CL Loss":    [f"{v:.4f}" for v in cl_hist],
        "Total Loss": [f"{v:.4f}" for v in tot_hist],
    })
    print("\n  Contrastive fine-tuning — last 5 epochs:")
    display(cl_df.tail(5).reset_index(drop=True))

    # ── Step 9: Inference on test set ─────────────────────────────────────────
    public_test_path   = "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/public_test/public_test.zh"
    private_test_path  = "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/private_test/private_test.zh"
    public_output_csv  = "./public_test_translations.csv"
    private_output_csv = "./private_test_translations.csv"

    final_ckpt_path = os.path.join(cfg.contrastive_dir, "best_contrastive_model_zh_vi.pth")
    if not os.path.isfile(final_ckpt_path):
        print("[WARN] Contrastive best model not found; using best translation model for inference.")
        final_ckpt_path = best_path

    print(f"\n[STEP 9] Inference — loading model from {final_ckpt_path} ...")
    inf_model, inf_sp, inf_cfg = load_model_from_checkpoint(final_ckpt_path, cfg.device)

    if os.path.isfile(public_test_path):
        pub_df = translate_file(
            inf_model, inf_sp, inf_cfg,
            public_test_path, public_output_csv,
            direction_token="<2vi>", split_name="Public",
        )
        show_sample_translations(pub_df, n=10)
    else:
        print(f"[WARN] Public test file not found: {public_test_path}")

    if os.path.isfile(private_test_path):
        priv_df = translate_file(
            inf_model, inf_sp, inf_cfg,
            private_test_path, private_output_csv,
            direction_token="<2vi>", split_name="Private",
        )
        show_sample_translations(priv_df, n=10)
    else:
        print(f"[WARN] Private test file not found: {private_test_path}")

    print("\n" + "=" * 70)
    print("  Pipeline complete.")
    print(f"  Plots            → {cfg.plot_dir}/")
    print(f"  Best model       → {best_path}")
    print(f"  CL best model    → {os.path.join(cfg.contrastive_dir, 'best_contrastive_model_zh_vi.pth')}")
    if os.path.isfile(public_output_csv):
        print(f"  Public CSV       → {public_output_csv}")
    if os.path.isfile(private_output_csv):
        print(f"  Private CSV      → {private_output_csv}")
    print("=" * 70)


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()
else:
    # When run inside a notebook, call main() directly
    main()


  ZH-VI Transformer — Full Pipeline
  Device : cuda

[STEP 1] Loading raw data ...
  Loaded 32,061 sentence pairs.
  ZH  word-length: mean=7.4  max=40
  VI  word-length: mean=8.2  max=48

[STEP 2] Training BPE tokenizer ...


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./tokenizer_train32/tmp_corpus.txt --model_prefix=./tokenizer_train32/spm_zh_vi_joint --vocab_size=8000 --model_type=bpe --character_coverage=1.0 --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3 --user_defined_symbols=<2zh>,<2vi>
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./tokenizer_train32/tmp_corpus.txt
  input_format: 
  model_prefix: ./tokenizer_train32/spm_zh_vi_joint
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <2zh

  Tokenizer saved → ./tokenizer_train32/spm_zh_vi_joint.model
  Vocabulary size: 8,000

[STEP 3] Filtering sentences (max 32 tokens) ...


pe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=20 min_freq=10
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=20 size=3420 all=31801 active=1582 piece=▁酒吧
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=20 size=3440 all=31783 active=1564 piece=▁服务员
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=19 size=3460 all=31763 active=1544 piece=Di
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=19 size=3480 all=31822 active=1603 piece=▁驾
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=19 size=3500 all=31858 active=1639 piece=▁单人
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=19 min_freq=10
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=19 size=3520 all=31845 active=1577 piece=▁继续
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=19 size=3540 all=31832 active=1564 piece=▁美国人
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=18 size=3560 all=31854 active=1586 piece=ơm
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=18 size=3580 all=31905 active=1637 piece

  Kept 31,874 / 32,061 pairs.
[PLOT] Saved length distribution → ./plots/length_distribution.png

[STEP 4] Building datasets ...
  Train pairs: 30,280   (zh→vi: 15,140  vi→zh: 15,140)
  Valid pairs: 797

[STEP 5] Building model ...
  Parameters: 157,179,200
  Architecture: 8 encoder / 8 decoder layers
  Attention: GQA (12 Q-heads / 4 KV-heads) + RoPE
  FFN: SwiGLU  (768 → 6144 → 3072 → 768)

[STEP 6] Training for 40 epochs ...


...


[CL Epoch 020/20]  CE=0.1569  CL=0.0141  Total=0.1583  lr=7.26e-06
  ✔ CL checkpoint saved → ./checkpoints_contrastive/cl_checkpoint_epoch_0020.pt

[Contrastive fine-tuning complete]  best_total_loss=0.1580
[PLOT] Saved contrastive curves → ./plots/contrastive_curves.png

  Contrastive fine-tuning — last 5 epochs:


,Epoch,CE Loss,CL Loss,Total Loss
0,16,0.1578,0.0164,0.1594
1,17,0.1576,0.0159,0.1592
2,18,0.1564,0.0153,0.1580
3,19,0.1569,0.0149,0.1584
4,20,0.1569,0.0141,0.1583



[STEP 9] Inference — loading model from ./checkpoints_contrastive/best_contrastive_model_zh_vi.pth ...


Translating [Public]:  24%|██▍       | 429/1781 [07:55<24:59,  1.11s/it]


KeyboardInterrupt: 

In [27]:
# =============================================================================
# SECTION 12 — INFERENCE & TEST OUTPUT
# =============================================================================

def translate_file(
    model:        TransformerModel,
    sp:           spm.SentencePieceProcessor,
    cfg:          TranslationConfig,
    input_path:   str,
    output_path:  str,
    direction_token: str = "<2vi>",
    split_name:   str = "Test",
    max_sentences: int = 20,
 ) -> pd.DataFrame:
    """Translate up to max_sentences lines from input_path and save to output_path (CSV)."""
    sentences = read_lines(input_path)[:max_sentences]
    rows      = []
    model.eval()

    if len(sentences) < max_sentences:
        print(f"[INFERENCE] Only found {len(sentences)} sentences in {split_name} input.")
    else:
        print(f"[INFERENCE] Translating first {max_sentences} sentences from {split_name} input.")

    for sentence in tqdm(sentences, desc=f"Translating [{split_name}]"):
        src_text   = f"{direction_token} {sentence}".strip()
        src_ids    = sp.encode(src_text, out_type=int)[: cfg.max_len]
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=cfg.device).unsqueeze(0)

        translation = beam_search_decode(
            model, src_tensor, sp, cfg,
            beam_size=cfg.beam_size, top_k=cfg.top_k,
            max_len=cfg.max_len, length_penalty=cfg.length_penalty,
        )[0]
        rows.append({"source_zh": sentence, "translation_vi": translation})

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"[INFERENCE] Saved {len(df)} translations → {output_path}")
    return df


def show_sample_translations(df: pd.DataFrame, n: int = 10) -> None:
    print(f"\n{'='*70}")
    print(f"  SAMPLE TEST TRANSLATIONS (first {n} rows)")
    print(f"{'='*70}")
    sample = df.head(n)
    for i, row in sample.iterrows():
        print(f"\n  [{i+1:02d}] ZH : {row['source_zh']}")
        print(f"       VI : {row['translation_vi']}")
    print(f"\n{'='*70}")

print("✅ Inference helpers defined.")


✅ Inference helpers defined.


In [29]:
# =============================================================================
# Cell 16 — Step 9: Inference on Test Set
# =============================================================================

def run_step9_inference(
    cfg: TranslationConfig,
    best_path: str,
    max_sentences: int = 20,
 ) -> None:
    final_ckpt_path = os.path.join(cfg.contrastive_dir, "best_contrastive_model_zh_vi.pth")
    if not os.path.isfile(final_ckpt_path):
        print("[WARN] Contrastive best model not found; using best translation model for inference.")
        final_ckpt_path = best_path

    print(f"\n[STEP 9] Inference — loading model from {final_ckpt_path} ...")
    inf_model, inf_sp, inf_cfg = load_model_from_checkpoint(final_ckpt_path, cfg.device)

    test_sets = [
        ("Public", "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/public_test/public_test.zh", "./public_test_translations.csv"),
        ("Private", "/kaggle/input/datasets/stardng/zh-vi-machine-translation/dataset/private_test/private_test.zh", "./private_test_translations.csv"),
    ]

    for split_name, input_path, output_path in test_sets:
        if os.path.isfile(input_path):
            df = translate_file(
                inf_model, inf_sp, inf_cfg,
                input_path, output_path,
                direction_token="<2vi>", split_name=split_name,
                max_sentences=max_sentences,
            )
            show_sample_translations(df, n=max_sentences)
        else:
            print(f"[WARN] {split_name} test file not found: {input_path}")

print("✅ Step 9 inference helper defined.")


✅ Step 9 inference helper defined.


In [33]:
# =============================================================================
# Cell 16 — Step 9: Inference on Test Set
# =============================================================================

# Chạy inference và in ra 20 câu cho mỗi file
run_step9_inference(
    cfg=cfg,
    best_path="/kaggle/working/checkpoints_contrastive/best_contrastive_model_zh_vi.pth",
    max_sentences=20,
)


[STEP 9] Inference — loading model from ./checkpoints_contrastive/best_contrastive_model_zh_vi.pth ...
[INFERENCE] Translating first 20 sentences from Public input.


Translating [Public]: 100%|██████████| 20/20 [00:24<00:00,  1.20s/it]


[INFERENCE] Saved 20 translations → ./public_test_translations.csv

  SAMPLE TEST TRANSLATIONS (first 20 rows)

  [01] ZH : 就 我 来说 , 通常 都 是 经营 的 问题 ， 很 少 带来 欢乐 。
       VI : Về về việc này , yên_tâm đều là kinh_doanh , rất ít_già để đối_với tôi .

  [02] ZH : 你 有 哪些 运动 器材 ？
       VI : Bạn có những bộ com_lê gì ?

  [03] ZH : 酒店 有 会议 设施 吗 ？
       VI : Có bất_kỳ khách_sạn nào vào ban_đêm không ?

  [04] ZH : 我 会 度 过 这 段 时间 。
       VI : Tôi sẽ ở lại qua thời_gian này .

  [05] ZH : 我 需要 一 张 填表 谢谢 。
       VI : Làm_ơn cần tôi điền vào giấy_phép .

  [06] ZH : 我 有 慢性 哮喘 。
       VI : Tôi bị mắc bệnh hen_suyễn .

  [07] ZH : 坐 船 过 池塘 要 多久 ？
       VI : Chúng_ta nên đi qua bao_lâu nữa từ đó ?

  [08] ZH : 她 想 要 一 杯 啤酒 。
       VI : Cô ấy sẽ uống một ly bia .

  [09] ZH : 去 纽约 。
       VI : Làm_ơn đến New_York .

  [10] ZH : 你 今晚 八点半 左右 不 是 偶然 在 十二 G 街 的 拐角 是 吗 ？
       VI : Bạn không nên nghỉ_ngơi khoảng 9 giờ 30 vào đêm nay phải không ?

  [11] ZH : 你 可以 帮 我 找找 我 的 手提箱 吗 ？
       VI : Bạn

Translating [Private]: 100%|██████████| 20/20 [00:19<00:00,  1.01it/s]

[INFERENCE] Saved 20 translations → ./private_test_translations.csv

  SAMPLE TEST TRANSLATIONS (first 20 rows)

  [01] ZH : 我 会 呆 两 天 。
       VI : Tôi sẽ ở lại trong hai ngày .

  [02] ZH : 我 现在 在 机场 。
       VI : Bây_giờ tôi đang ở sân_bay .

  [03] ZH : 玛丽 不 比 亨利 大 。
       VI : Mary không lớn bằng văn_phòng .

  [04] ZH : 这些 问题 并不 新鲜 。
       VI : Những vấn_đề này không_thể phù_hợp .

  [05] ZH : 再 说 一 次 你 的 姓名 。
       VI : Nói lại lần nữa nói_chuyện với bạn .

  [06] ZH : 我 大概 有 三千 美元 。
       VI : Tôi có khoảng 3000 đô_la .

  [07] ZH : 当然 了 先生 , 可是 你 出 什么 事 了 ？
       VI : Chắc_chắn rồi , nhưng anh ra có chuyện gì ?

  [08] ZH : 你 需要 活动 。
       VI : Bạn cần giấy_phép .

  [09] ZH : 它 在 星期几 开 ？
       VI : Nó ở mà nó mở vào thứ mấy ?

  [10] ZH : 你 知道 日本 的 摔跤 传统 , 在 相扑 冠军 之间 有 一 个 是 美国人 吗 ？
       VI : Bạn biết rõ về truyền_hình Úc , ở Nhật_Bản được một trong những người Nhật_Bản phải không ?

  [11] ZH : 这 是 什么 宝石 ？
       VI : Nó là loại dốc gì ?

  [12] ZH : 就 一 个 包 吗 ？
   